# Proyecto SQL

Una startup busca expandirse en el mercado mediante la incorporación de innovaciones en su servicio.
El objetivo de este estudio es analizar y explorar los datos disponibles para identificar patrones, tendencias y oportunidades dentro del mercado de libros, con el fin de proponer una posible propuesta de valor para el desarrollo de un nuevo producto.

### Conexión a la base de datos

In [2]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

### BOOKS

In [3]:
query = """  
SELECT
    *
FROM
    books
LIMIT 5;     -- Imprimir las primeras filas
"""
pd.read_sql(query, con=engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Contiene datos sobre los libros como el título, número de páginas, año de publicación y el ID del author. Esta tabla se puede relacionar con todas las demás, books contiene datos que pueden ser utilizados para complementar la información de otras tablas o viseversa.

### AUTHORS

In [4]:
query = """  
SELECT
    *
FROM
    authors
LIMIT 5;     -- Imprimir las primeras filas
"""
pd.read_sql(query, con=engine)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Esta tabla contiene el ID y nombre del author de los libros que se encuentra en la tabla **books**, es de gran utilidad si queremos identificar a los autores más reconocidos, la cantidad de libros que tienen.

### PUBLISHERS

In [5]:
query = """  
SELECT
    *
FROM
    publishers
LIMIT 5;     -- Imprimir las primeras filas
"""
pd.read_sql(query, con=engine)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Contiene las editoriales existentes, la cuál se conecta con la tabla **books** y con ella se puede identificar que libros han sido publicados por una editorial en específico.

### RATINGS

In [6]:
query = """  
SELECT
    *
FROM
    ratings
LIMIT 5;     -- Imprimir las primeras filas
"""
pd.read_sql(query, con=engine)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Contiene información sobre las calificaciones dadas a los libros por los usuarios, esta se puede relacionar con la tabla **books** y la tabla **reviews**. Ayuda a identificar entre los libros que aman y odian entre los lectores.

### REVIEWS

In [7]:
query = """  
SELECT
    *
FROM
    reviews
LIMIT 5;     -- Imprimir las primeras filas
"""
pd.read_sql(query, con=engine)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


Contiene las reseñas hechas por usuarios que han leído los libros que se encuentran en la tabla **books**, en donde explican su punto de vista o recomendaciones al terminar un libro. No todos los usuarios dejan una reseña en texto. también es posible conectarla con la tabla **reviews**.

### Número de libros publicados después del 1 de enero de 2000.

In [8]:
query = """
SELECT
    COUNT(book_id) AS total_books
FROM 
    books 
WHERE 
    publication_date > '2000-01-01'
"""
pd.read_sql(query, con=engine)

,total_books
0,819


Se identificó que 819 libros del dataset fueron publicados después del 1 de enero del 2000, lo que indica que una parte significativa del catálogo corresponde a publicaciones relativamente recientes.
Esto sugiere que la plataforma cuenta con un contenido actualizado y relevante, lo cual es importante para mantener el interés de los usuarios y reflejar tendencias editoriales contemporáneas.

### Número de reseñas de usuarios y la calificación promedio para cada libro.

In [9]:
query = """
SELECT
    books.book_id,
    books.title,
    ratings.avg_rating,
    reviews.review_count
FROM books 
LEFT JOIN (
    SELECT
        book_id,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
) ratings ON ratings.book_id = books.book_id
LEFT JOIN (
    SELECT
        book_id,
        COUNT(review_id) AS review_count
    FROM reviews
    GROUP BY book_id
) reviews ON reviews.book_id = books.book_id;
"""
pd.read_sql(query, con=engine)

,book_id,title,avg_rating,review_count
0,652,The Body in the Library (Miss Marple #3),4.500000,2.0
1,273,Galápagos,4.500000,2.0
2,51,A Tree Grows in Brooklyn,4.250000,5.0
3,951,Undaunted Courage: The Pioneering First Missio...,4.000000,2.0
4,839,The Prophet,4.285714,4.0
...,...,...,...,...
995,64,Alice in Wonderland,4.230769,4.0
996,55,A Woman of Substance (Emma Harte Saga #1),5.000000,2.0
997,148,Christine,3.428571,3.0
998,790,The Magicians' Guild (Black Magician Trilogy #1),3.500000,2.0


El análisis de las reseñas y calificaciones muestra que no todos los libros reciben el mismo nivel de interacción por parte de los usuarios. Algunos títulos concentran una mayor cantidad de reseñas y calificaciones, mientras que otros presentan una participación limitada.
Esto sugiere que la visibilidad, popularidad o relevancia del libro influyen directamente en el nivel de participación de los usuarios dentro de la plataforma.

### Editorial que ha publicado el mayor número de libros con más de 50 páginas (esto ayudará a excluir folletos y publicaciones similares del análisis).

In [10]:
query = """
SELECT 
    publishers.publisher_id, 
    publishers.publisher, 
    COUNT(books.book_id) AS books
FROM 
    books 
    INNER JOIN publishers ON publishers.publisher_id = books.publisher_id 
WHERE 
    num_pages > 50
GROUP BY 
    publishers.publisher, publishers.publisher_id 
ORDER BY 
    books DESC
LIMIT 1;
"""
pd.read_sql(query, con=engine)

,publisher_id,publisher,books
0,212,Penguin Books,42


Se identificó que Penguin Books es la editorial que ha publicado el mayor número de libros (considerando únicamente libros con más de 50 páginas).

Esto sugiere que Penguin Books tiene una presencia dominante en el catálogo, lo que podría indicar una estrategia editorial enfocada en volumen y diversidad de títulos. Para análisis comerciales, esta editorial representa un actor clave dentro del mercado editorial.

### Autor que tiene la más alta calificación promedio del libro: solo los libros con al menos 50 calificaciones.

In [11]:
query = """
SELECT 
    authors.author
FROM books 
INNER JOIN ( 
    SELECT 
        book_id, 
        AVG(rating) AS avg_rating, 
        COUNT(rating) AS count_rating 
    FROM 
        ratings 
    GROUP BY 
        book_id 
    HAVING 
        COUNT(rating) >= 50 
) ratings ON ratings.book_id = books.book_id 
INNER JOIN authors ON authors.author_id = books.author_id 
GROUP BY 
    authors.author
ORDER BY 
    AVG(ratings.avg_rating) DESC 
LIMIT 1;
"""
pd.read_sql(query, con=engine)

,author
0,J.K. Rowling/Mary GrandPré


El análisis mostró que J.K. Rowling / Mary GrandPré es el autor con la calificación promedio más alta, considerando solo libros con al menos 50 calificaciones.

Este resultado indica que el autor no solo tiene libros bien valorados, sino que dichas valoraciones están respaldadas por un alto número de opiniones, lo que hace que la calificación sea estadísticamente confiable. Esto refuerza su relevancia y aceptación sostenida entre los lectores.

### Número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

In [14]:
query = """
SELECT
    AVG(count_reviews) AS avg_reviews
FROM (
    SELECT
        reviews.username,
        COUNT(reviews.review_id) AS count_reviews
    FROM 
        reviews
        INNER JOIN (
            SELECT
                username,
                COUNT(rating) AS count_rating
            FROM 
                ratings
            GROUP BY
                username
            HAVING 
                COUNT(rating) > 50
        ) ratings ON ratings.username = reviews.username 
    GROUP BY 
        reviews.username
) avg_reviews;
"""
pd.read_sql(query, con=engine)

,avg_reviews
0,24.333333


Se encontró que los usuarios que calificaron más de 50 libros escribieron en promedio 24.33 reseñas de texto.

Aunque estos usuarios son altamente activos en términos de calificaciones, no todos acompañan sus valoraciones con reseñas escritas. Esto sugiere que escribir una reseña requiere un mayor nivel de compromiso que simplemente calificar, y que solo una parte de los usuarios más activos aporta contenido textual detallado.

## Conclusión general

A partir del análisis de los datos, se identificaron distintos patrones relevantes dentro del mercado de libros. Se observó que una parte considerable del catálogo corresponde a publicaciones recientes, lo que indica un enfoque hacia contenido actualizado. Asimismo, se detectó que ciertas editoriales, como Penguin Books, concentran una mayor producción de títulos, lo que sugiere su fuerte presencia y competitividad en el mercado.

En cuanto a los autores, se identificó que algunos presentan calificaciones promedio significativamente más altas, destacando a J.K. Rowling/Mary GrandPré, lo cual refleja una alta aceptación por parte de los usuarios. Además, el análisis del comportamiento de los usuarios mostró que aquellos con mayor actividad de calificación también tienden a generar una cantidad considerable de reseñas de texto, lo que evidencia un alto nivel de participación dentro de la plataforma.

En conjunto, estos hallazgos permiten identificar oportunidades para el desarrollo de un nuevo producto enfocado en contenido popular, autores altamente valorados y usuarios con alto nivel de interacción, lo cual podría ser aprovechado por una startup para diferenciar su propuesta de valor en el mercado editorial.